<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-06-function-calling/lesson-6.3-parallel-tools/notebooks/GCP_Capstone_6.3_ParallelTools.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6.3 Parallel Calls & Built-in Tools — Google Search + Code Exec + Custom
**Netsetos GenAI Engineering — GCP Capstone**

Execute parallel calls concurrently. Combine Google Search, Code Execution, and custom functions.


## Setup


In [ ]:
USD_INR = 85          # course-wide conversion rate
RATES = {"standard": 0.05, "priority": 0.12, "bulk": 0.03}

# DocuMind's demo corpus - the SAME five documents in every
# lesson of Modules 6, 7 and 8, so results stay comparable.
# 119 pages in total, which is what the cost examples bill.
GS = "gs://documind-acme"
_DOCS = [
    # doc_id, doc_type, page, score, quote
    ("hr_policy_2026", "policy", 12, 0.94,
     "A senior engineer serves a notice period of 60 days."),
    ("hr_policy_2026", "policy", 31, 0.81,
     "Earned leave is encashed on exit, capped at 45 days."),
    ("msa_acme_2026", "contract", 8, 0.88,
     "Either party may terminate on 90 days written notice."),
    ("inv_2026_0412", "invoice", 1, 0.76,
     "Total payable Rs 1,84,500, inclusive of 18% GST."),
    ("gstr1_q1_fy27", "form", 4, 0.68,
     "Outward taxable supplies for the quarter, GSTR-1."),
    ("rag_survey_2026", "research_paper", 6, 0.72,
     "Hybrid retrieval mixes dense and sparse signals."),
]
CORPUS = [{"chunk_id": f"{d}#{p}", "doc_type": t, "page": p,
           "source_uri": f"{GS}/{d}.pdf", "quote": q,
           "score": s} for d, t, p, s, q in _DOCS]
CITATION_FIELDS = ("chunk_id", "source_uri", "page", "quote",
                   "score")

# Document lengths, for the tool-chaining demos: retrieve
# finds chunks, and the cost tool bills whole documents.
DOC_PAGES = {"hr_policy_2026": 48, "msa_acme_2026": 32,
             "inv_2026_0412": 3, "gstr1_q1_fy27": 12,
             "rag_survey_2026": 24}          # 119 pages


def docs_of(citations: list) -> dict:
    """Distinct source documents behind a set of citations."""
    return {c["chunk_id"].split("#")[0]: True
            for c in citations}

!pip install -q google-genai

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID,
                      location='global')

# The same three tools as 6.1 and 6.2 - one definition, one signature
USD_INR = 85


def retrieve(query: str, doc_type: str = "all",
             top_k: int = 5) -> dict:
    """Retrieve grounded passages from DocuMind's corpus.

    Args:
        query: The question, in natural language
        doc_type: policy, contract, invoice, form,
            research_paper, or all
        top_k: How many passages to return
    """
    # A mock, but not a stub: it really filters and ranks, so
    # a question the corpus cannot answer returns NOTHING and
    # answerable=False. A mock that always succeeds teaches
    # that retrieval always succeeds - the one thing it never
    # does.
    words = {w for w in query.lower().split() if len(w) > 3}
    hits = [c for c in CORPUS
            if doc_type in ("all", c["doc_type"])
            and any(w in c["quote"].lower() for w in words)]
    hits.sort(key=lambda c: -c["score"])
    hits = hits[:top_k]
    top = hits[0]["score"] if hits else 0.0
    return {
        "citations": [{k: c[k] for k in CITATION_FIELDS}
                      for c in hits],
        "answerable": bool(hits),
        "confidence": ("high" if top >= 0.85 else
                       "medium" if hits else "low"),
    }

def calculate_processing_cost(
        total_pages: int, num_documents: int = 1,
        processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents
        num_documents: How many documents those pages span
        processing_type: standard, priority, or bulk
    """
    rate = RATES.get(processing_type, RATES["standard"])
    cost = total_pages * rate
    return {"num_documents": num_documents,
            "total_pages": total_pages,
            "processing_type": processing_type,
            "rate_per_page": rate,
            "cost_usd": round(cost, 2),
            "cost_inr": round(cost * USD_INR, 2)}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get DocuMind pipeline usage statistics.

    Args:
        metric: queries, costs, latency, or users
        days: Number of days to look back
    """
    mock = {"queries": 1247, "costs": 18.50,
            "latency": 245, "users": 42}
    return {"metric": metric, "period": f"last {days} days",
            "value": mock.get(metric, 0), "trend": "+12%"}

TOOLS = [retrieve, calculate_processing_cost, get_usage_stats]
print('Ready')


## Cell 1: Google Search Grounding


In [ ]:
# Google Search: model queries the web automatically
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What are the latest data protection regulations in India for 2026?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())])
)
print('=== Google Search Grounded Response ===')
print(response.text[:300])

# Check grounding metadata
metadata = response.candidates[0].grounding_metadata
if metadata:
    print(f'\nSearch queries: {metadata.web_search_queries}')
    if metadata.grounding_chunks:
        for chunk in metadata.grounding_chunks[:3]:
            print(f'  Source: {chunk.web.title}')
            print(f'  URL: {chunk.web.uri}')


## Cell 2: Code Execution


In [ ]:
# Code Execution: Gemini generates + runs Python
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Calculate compound interest on Rs 10,00,000 at 8.5% '
             'for 5 years compounded quarterly. Show the formula and result.',
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)])
)

print('=== Code Execution Response ===')
for part in response.candidates[0].content.parts:
    if part.executable_code:
        print(f'Generated code:\n{part.executable_code.code}\n')
    if part.code_execution_result:
        print(f'Execution result: {part.code_execution_result.output}')
    if part.text:
        print(f'Explanation: {part.text[:200]}')


## Cell 3: Parallel Function Calls


In [ ]:
# Trigger parallel calls with an independent multi-part query
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Search for our legal documents AND show this week query stats',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
)

print('=== Parallel Calls ===')
if response.function_calls:
    print(f'Number of calls: {len(response.function_calls)}')
    for fc in response.function_calls:
        print(f'  {fc.name}({dict(fc.args)}) id={fc.id}')
else:
    print('No function calls (model responded with text)')


## Cell 4: Concurrent Execution


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def execute_parallel_sync(function_calls, functions):
    """Execute multiple function calls concurrently."""
    start = time.time()
    results = [None] * len(function_calls)
    with ThreadPoolExecutor(max_workers=len(function_calls)) as pool:
        futures = {
            pool.submit(functions[fc.name], **fc.args): i
            for i, fc in enumerate(function_calls)
        }
        for future in futures:
            i = futures[future]
            fc = function_calls[i]
            try:
                result = future.result(timeout=15)
                results[i] = types.Part.from_function_response(
                    name=fc.name, response={'result': result})
            except Exception as e:
                results[i] = types.Part.from_function_response(
                    name=fc.name, response={'error': str(e)})
    elapsed = time.time() - start
    print(f'Parallel execution: {elapsed:.2f}s for {len(function_calls)} calls')
    return results

# Test with the parallel calls from Cell 3
if response.function_calls and len(response.function_calls) > 1:
    FUNCTIONS = {'retrieve': retrieve,
                 'calculate_processing_cost': calculate_processing_cost,
                 'get_usage_stats': get_usage_stats}
    result_parts = execute_parallel_sync(
        list(response.function_calls), FUNCTIONS)
    print(f'Got {len(result_parts)} results')
    for rp in result_parts:
        print(f'  {rp.function_response.name}: {rp.function_response.response}')


## Cell 5: Google Search + Custom Functions Combined


In [ ]:
# NOTE: On Vertex AI a built-in tool (Google Search) and custom function tools
# cannot be combined in ONE request -- tool "context circulation"
# (include_server_side_tool_invocations) is Gemini-Developer-API only, not Vertex.
# Route each query to the appropriate SINGLE tool set instead.
SEARCH_TOOL = types.Tool(google_search=types.GoogleSearch())
DOC_TOOL = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name='retrieve',
        description='Search internal company documents by keyword.',
        parameters={'type': 'object',
                    'properties': {'query': {'type': 'string'},
                                   'doc_type': {'type': 'string', 'enum': ['legal', 'invoice', 'all']}},
                    'required': ['query']})])

# Internal question -> custom document tool (disable auto-calling to see the routing)
r1 = client.models.generate_content(
    model='gemini-3.6-flash', contents='Find our privacy policy',
    config=types.GenerateContentConfig(
        tools=[DOC_TOOL],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)))
print('Internal ->', r1.function_calls[0].name if r1.function_calls else (r1.text or '')[:150])

# External question -> Google Search
r2 = client.models.generate_content(
    model='gemini-3.6-flash', contents='What are the latest GDPR updates?',
    config=types.GenerateContentConfig(tools=[SEARCH_TOOL]))
print('External ->', (r2.text or '')[:150])

## Cell 6: Code Execution + Custom Functions


In [ ]:
# NOTE: On Vertex AI, custom function tools and a built-in tool (Code Execution)
# run in SEPARATE requests -- they can't be combined in one call. Chain them at
# the app level: fetch data with the custom tool, then compute with Code Execution.
# Step 1: get the standard per-page cost via the custom cost function.
r1 = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='How much would it cost to process 500 pages at standard rate?',
    config=types.GenerateContentConfig(tools=TOOLS))
print('Data (custom tools):', (r1.text or '')[:200])
# Step 2: compute the weekly -> monthly projection with Code Execution.
r2 = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='If that 500-page job runs once every week, compute the total monthly cost. Show the calculation.',
    config=types.GenerateContentConfig(tools=[types.Tool(code_execution=types.ToolCodeExecution)]))
print('Compute (code execution):', (r2.text or '')[:300])

## Cell 7: Multi-Tool DocuMind Agent


In [ ]:
# Complete multi-tool agent. On Vertex AI, built-in tools (Google Search, Code
# Execution) and custom function tools cannot share ONE request, so we route each
# query to a SINGLE tool set with a lightweight keyword classifier.
SYSTEM_PROMPT = '''You are DocuMind AI, a document intelligence assistant.
Use the single most appropriate tool for the query.'''

def route_tools(query):
    q = query.lower()
    if any(w in q for w in ['calculate','tax','compute','%','average','difference','statistic']):
        return [types.Tool(code_execution=types.ToolCodeExecution)], 'code_execution'
    if any(w in q for w in ['current','latest','news','act','regulation','benchmark','industry']):
        return [types.Tool(google_search=types.GoogleSearch())], 'google_search'
    if any(w in q for w in ['find','document','legal','cost','usage','stats','process','invoice']):
        return list(TOOLS), 'custom_functions'
    return [], 'text (no tool)'

def answer_with_routing(query):
    tools, label = route_tools(query)
    cfg = (types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, tools=tools)
           if tools else types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT))
    return client.models.generate_content(model='gemini-3.6-flash', contents=query, config=cfg), label

test_queries = [
    'Find our legal documents',                # custom
    'What is the current DPDP Act in India?',  # Google Search
    'Calculate 15% tax on Rs 2,50,000',        # Code Execution
    'Hello, how can you help me?',             # no tool
]
for q in test_queries:
    r, label = answer_with_routing(q)
    print(f'Q: {q}')
    print(f'  Routed to: {label}')
    print(f'  A: {(r.text or "[function call]")[:100]}\n')

## ✅ Lesson 6.3 Complete! MODULE 6 COMPLETE!

**Parallel execution mastered:**
- ✅ asyncio.gather and ThreadPoolExecutor patterns
- ✅ ID-based result correlation
- ✅ Partial failure handling (1:1 parity)

**Built-in tools mastered:**
- ✅ Google Search grounding with metadata + citations
- ✅ Code Execution sandbox (numpy, pandas, matplotlib)
- ✅ Multi-tool routing: one tool family per call on Vertex (combining them in one request is Developer-API-only, and Preview)
- ✅ System instruction routing for tool selection

**Module 6 Complete — 3 Lessons:**
- 6.1: FunctionDeclarations, AUTO/ANY/NONE modes
- 6.2: While-loop dispatcher, sequential chaining, FunctionRegistry
- 6.3: Parallel execution, Google Search + Code Exec + custom tools

**Next: Module 7 — MCP Servers on Cloud Run**
